# HPC Course: From Linux Basics to Production Workflows
## University of Geneva — Bamboo Cluster

**A semester-long, hands-on course designed to run interactively on the HPC.**

This notebook is meant to be executed directly on the Bamboo cluster via Jupyter or OpenOnDemand.
Each module contains theory, live commands, and exercises. Commands prefixed with `!` run in the shell.

> **Cluster**: Bamboo (`login1.bamboo.hpc.unige.ch`) — Rocky Linux 9, SLURM scheduler, BeeGFS storage

---

### Course Outline

| # | Module | Key Topics |
|---|--------|-----------|
| 1 | Linux Fundamentals | Navigation, files, permissions, pipes, text processing |
| 2 | Shell Scripting | Variables, loops, conditionals, functions, scripts |
| 3 | Connecting to the HPC | SSH, keys, config, file transfer, X2Go |
| 4 | Cluster Architecture | Nodes, partitions, CPUs, GPUs, InfiniBand |
| 5 | Storage Systems | Home, scratch, fast, local, quotas, retention |
| 6 | Environment & Modules | `module` / `lmod`, toolchains, conda, containers |
| 7 | SLURM — Basics | `sbatch`, `salloc`, `srun`, partitions, resources |
| 8 | SLURM — Job Arrays | Embarrassingly parallel, array indexing, throttling |
| 9 | SLURM — Advanced | Dependencies, checkpointing, priority, backfill |
| 10 | MPI & Parallel Computing | Distributed jobs, `mpiexec`, `ntasks`, conda-pack |
| 11 | GPU Computing | GPU partitions, CUDA, `nvidia-smi`, GPU job scripts |
| 12 | Monitoring & Debugging | `squeue`, `sacct`, `seff`, community forum issues |
| 13 | Data Management | rsync, rclone, SWITCHfilesender, archiving |
| 14 | Best Practices | Resource estimation, green HPC, common mistakes |
| 15 | Real Workflows | NoisePy, MANgOSTA, BayHunter, dispersion picking |
| 16 | Putting It All Together | End-to-end project: data → processing → results |

### Resources

- [Official HPC Documentation](https://doc.eresearch.unige.ch/hpc/toc)
- [HPC Community Forum](https://hpc-community.unige.ch/) — report issues, request software, search for solutions
- HPC Support email: `hpc@unige.ch`

### Upcoming Maintenance Windows (check forum for updates)
Clusters have regular maintenance windows (typically 3-4 days). Plan your submissions accordingly:
- Check announcements at: https://hpc-community.unige.ch/c/hpc-announce/

---
# Module 1: Linux Fundamentals
---

The HPC cluster runs **Rocky Linux 9** (RHEL-family). All interaction happens through the **command line** (terminal/shell). This module covers the essential commands you need every day.

## 1.1 The Shell

When you connect via SSH, you land in a **Bash** shell. The prompt shows your username, hostname, and current directory:

```
henrymi@login1:~$
```

- `~` = your home directory (`/home/users/h/henrymi`)
- `$` = regular user (vs `#` for root, which you won't have)

## 1.2 Navigating the Filesystem

| Command | Purpose | Example |
|---------|---------|---------|
| `pwd` | Print working directory | `pwd` → `/home/users/h/henrymi` |
| `ls` | List files | `ls -la` (detailed + hidden) |
| `cd` | Change directory | `cd scratch/` or `cd ~` (home) |
| `mkdir` | Create directory | `mkdir -p project/data` |
| `tree` | Directory tree view | `tree -L 2 project/` |

### Absolute vs Relative Paths
- **Absolute**: starts with `/` → `/home/users/h/henrymi/project`
- **Relative**: from current dir → `../project` (up one level, then into project)

### Special Directories
- `.` = current directory
- `..` = parent directory
- `~` = home directory
- `-` = previous directory (`cd -`)

In [ ]:
# Practice 1.1: Navigate the filesystem
# Run these commands and observe the output

!echo "Current directory:" && pwd
!echo "---"
!echo "Home directory contents:" && ls -la ~/ | head -20
!echo "---"
!echo "Disk usage of home:" && du -sh ~/ 2>/dev/null || echo "(may take a moment)"
!echo "---"
!echo "Who am I?" && whoami && hostname

## 1.3 Working with Files

| Command | Purpose | Example |
|---------|---------|---------|
| `cat` | Display file content | `cat file.txt` |
| `head` / `tail` | First/last N lines | `head -20 file.txt` |
| `less` | Page through file | `less file.txt` (q to quit) |
| `cp` | Copy | `cp file.txt backup.txt` |
| `mv` | Move/rename | `mv old.txt new.txt` |
| `rm` | Remove | `rm file.txt` or `rm -r dir/` |
| `touch` | Create empty file | `touch newfile.txt` |
| `wc` | Count lines/words | `wc -l file.txt` |
| `diff` | Compare files | `diff file1.txt file2.txt` |

> **WARNING**: `rm` is permanent. There is no trash can on Linux. Always double-check before running `rm -r`.

In [ ]:
# Practice 1.2: File operations
# Create a temporary workspace for this course

!mkdir -p ~/hpc_course/module1 && cd ~/hpc_course/module1 && \
  echo "Hello HPC!" > hello.txt && \
  echo "This is line 2" >> hello.txt && \
  echo "--- File contents ---" && cat hello.txt && \
  echo "--- Word count ---" && wc hello.txt && \
  echo "--- File info ---" && ls -la hello.txt

## 1.4 Text Processing

These tools are the backbone of data wrangling on Linux:

| Command | Purpose | Example |
|---------|---------|---------|
| `grep` | Search for patterns | `grep "error" logfile.out` |
| `sed` | Stream editor (find/replace) | `sed 's/old/new/g' file.txt` |
| `awk` | Column-based processing | `awk '{print $1, $3}' data.csv` |
| `sort` | Sort lines | `sort -n numbers.txt` |
| `uniq` | Remove duplicates | `sort file \| uniq -c` |
| `cut` | Extract columns | `cut -d',' -f1,3 data.csv` |

## 1.5 Pipes and Redirection

The **pipe** (`|`) sends one command's output to another's input. This is what makes Linux powerful:

```bash
# Count unique error types in a log file
grep "ERROR" simulation.log | awk '{print $4}' | sort | uniq -c | sort -rn

# Find the 5 largest files in a directory
ls -lS data/ | head -5
```

### Redirection
- `>` — overwrite file: `echo "hello" > file.txt`
- `>>` — append to file: `echo "world" >> file.txt`
- `2>` — redirect errors: `command 2> errors.log`
- `2>&1` — merge stderr into stdout: `command > output.log 2>&1`

In [ ]:
# Practice 1.3: Pipes and text processing
# Generate some sample data and process it

!cd ~/hpc_course/module1 && \
  echo "station,latitude,longitude,elevation" > stations.csv && \
  echo "STA01,46.20,6.15,400" >> stations.csv && \
  echo "STA02,46.25,6.20,550" >> stations.csv && \
  echo "STA03,46.18,6.10,320" >> stations.csv && \
  echo "STA04,46.22,6.18,480" >> stations.csv && \
  echo "STA05,46.30,6.25,600" >> stations.csv && \
  echo "--- All stations ---" && cat stations.csv && \
  echo "--- Stations above 46.20 lat (using awk) ---" && \
  awk -F',' 'NR>1 && $2>46.20 {print $1, "lat="$2}' stations.csv && \
  echo "--- Sort by elevation (column 4, numeric) ---" && \
  tail -n +2 stations.csv | sort -t',' -k4 -n && \
  echo "--- Number of stations ---" && \
  tail -n +2 stations.csv | wc -l

## 1.6 Permissions

Every file has an owner, group, and permission bits:

```
-rwxr-xr--  1  henrymi  hpc_users  4096  Sep 21  script.sh
│├─┤├─┤├─┤
│ │   │  └── Others: read only
│ │   └───── Group: read + execute
│ └───────── Owner: read + write + execute
└──────────── File type (- = file, d = directory, l = symlink)
```

| Command | Purpose |
|---------|---------|
| `chmod 755 script.sh` | Owner: rwx, Group: r-x, Others: r-x |
| `chmod +x script.sh` | Add execute for all |
| `chmod u+w file.txt` | Add write for owner |

> **Note**: On the HPC, home directory permissions are fixed at `0700` (user-only) and reset daily. You cannot share files via permission changes — use shared directories instead.

In [ ]:
# Practice 1.4: Permissions
!cd ~/hpc_course/module1 && \
  echo '#!/bin/bash' > myscript.sh && \
  echo 'echo "Hello from $(hostname) at $(date)"' >> myscript.sh && \
  echo "--- Before chmod ---" && ls -la myscript.sh && \
  chmod +x myscript.sh && \
  echo "--- After chmod +x ---" && ls -la myscript.sh && \
  echo "--- Running it ---" && bash myscript.sh

## 1.7 Finding Things

| Command | Purpose | Example |
|---------|---------|---------|
| `find` | Find files by name/type/time | `find . -name "*.py" -mtime -7` |
| `which` | Locate a command | `which python` |
| `locate` | Fast filename search (if available) | `locate station.csv` |
| `grep -r` | Search file contents recursively | `grep -r "import obspy" ~/project/` |

> **IMPORTANT**: Never run `find` or `du` on `/srv/beegfs/scratch/` from the root — it will hammer the metadata server and slow the cluster for everyone. Always scope your searches to specific subdirectories.

In [ ]:
# Practice 1.5: Finding files
# Find all shell scripts in your home directory (excluding hidden dirs)
!find ~/ -maxdepth 3 -name "*.sh" -not -path "*/.*" 2>/dev/null | head -15
!echo "---"
# Find Python files modified in the last 7 days
!find ~/ -maxdepth 3 -name "*.py" -mtime -7 -not -path "*/.*" 2>/dev/null | head -10

---
# Module 2: Shell Scripting
---

Shell scripts are the glue of HPC workflows. Every SLURM job submission is a shell script. Mastering Bash scripting is essential.

## 2.1 Variables

```bash
# Assignment (no spaces around =)
NAME="henrymi"
NPROCS=20
DATA_DIR="/srv/beegfs/scratch/users/h/henrymi/project"

# Usage (with $)
echo "User: $NAME"
echo "Processing with $NPROCS cores"
echo "Data in: ${DATA_DIR}/raw"   # Braces for clarity

# Command substitution
TODAY=$(date +%Y-%m-%d)
NFILES=$(ls *.h5 | wc -l)
```

## 2.2 Special Variables

| Variable | Meaning |
|----------|---------|
| `$0` | Script name |
| `$1`, `$2`, ... | Positional arguments |
| `$#` | Number of arguments |
| `$@` | All arguments |
| `$?` | Exit code of last command |
| `$$` | Process ID |

## 2.3 Conditionals

```bash
if [ -f "data.h5" ]; then
    echo "File exists"
elif [ -d "data/" ]; then
    echo "Directory exists"
else
    echo "Nothing found"
fi

# Useful test operators:
# -f file    File exists
# -d dir     Directory exists
# -z "$var"  Variable is empty
# -n "$var"  Variable is not empty
# $a -eq $b  Numeric equality
# $a -gt $b  Greater than
```

## 2.4 Loops

```bash
# Loop over files
for f in *.h5; do
    echo "Processing: $f"
    python process.py "$f"
done

# Loop over numbers
for i in $(seq 1 10); do
    echo "Iteration $i"
done

# Loop over array
periods=(0.5 1.0 1.5 2.0 2.5)
for per in "${periods[@]}"; do
    echo "Period: $per s"
done

# While loop
count=0
while [ $count -lt 5 ]; do
    echo "Count: $count"
    count=$((count + 1))
done
```

In [ ]:
# Practice 2.1: Write a shell script
!cd ~/hpc_course && mkdir -p module2 && cat > module2/process_stations.sh << 'SCRIPT'
#!/bin/bash
# Process a list of stations from a CSV file
# Usage: bash process_stations.sh stations.csv

INPUT_FILE=${1:-"../module1/stations.csv"}

if [ ! -f "$INPUT_FILE" ]; then
    echo "ERROR: File $INPUT_FILE not found!"
    exit 1
fi

NSTA=$(tail -n +2 "$INPUT_FILE" | wc -l)
echo "Processing $NSTA stations from $INPUT_FILE"
echo "=========================================="

# Skip header, loop through stations
tail -n +2 "$INPUT_FILE" | while IFS=',' read -r sta lat lon elev; do
    echo "Station: $sta | Lat: $lat | Lon: $lon | Elev: ${elev}m"
    
    # Example: flag high-elevation stations
    if [ $(echo "$elev > 500" | bc -l 2>/dev/null || echo 0) -eq 1 ]; then
        echo "  → High elevation station!"
    fi
done

echo "=========================================="
echo "Done! Processed $NSTA stations."
SCRIPT

echo "--- Script contents ---"
cat module2/process_stations.sh
echo ""
echo "--- Running it ---"
bash module2/process_stations.sh module1/stations.csv

## 2.5 Arrays in Bash

Arrays are heavily used in SLURM array jobs (Module 8):

```bash
# Define array
periods=(0.9 1.0 1.1 1.2 1.3 1.4 1.5)

# Access elements
echo ${periods[0]}          # First element: 0.9
echo ${periods[@]}          # All elements
echo ${#periods[@]}         # Length: 7

# Use with SLURM array index
ARRAY_INDEX=$SLURM_ARRAY_TASK_ID
current_period=${periods[$ARRAY_INDEX]}
```

## 2.6 Functions

```bash
check_output() {
    local outdir=$1
    local expected=$2
    local actual=$(ls "$outdir"/*.h5 2>/dev/null | wc -l)
    
    if [ "$actual" -eq "$expected" ]; then
        echo "✓ $outdir: $actual/$expected files"
        return 0
    else
        echo "✗ $outdir: $actual/$expected files (MISSING!)"
        return 1
    fi
}

# Usage
check_output "/path/to/CCF" 100
```

---
# Module 3: Connecting to the HPC
---

## 3.1 SSH Connection

The Bamboo cluster is accessed via SSH:

```bash
ssh youruser@login1.bamboo.hpc.unige.ch
```

### SSH Config (recommended)

Add to `~/.ssh/config` on your **local** machine:

```
Host mibam
    HostName login1.bamboo.hpc.unige.ch
    User henrymi
    ForwardAgent yes
    
Host cpu*
    HostName %h
    User henrymi
    ProxyJump mibam
```

Then simply: `ssh mibam`

### SSH Keys

Generate a key pair (do this on your local machine):
```bash
ssh-keygen -t ed25519 -C "your.email@unige.ch"
```

Upload the **public** key to https://my-account.unige.ch (syncs every 5 minutes).

> **Security tip**: Never store passwords or tokens in plain text files on the cluster (like `.bashrc`). Use SSH keys and environment variables loaded at runtime.

## 3.2 Login Node Rules

The login node (`login1.bamboo`) has strict resource limits:
- **2 CPU cores, 8 GB RAM maximum**
- Use it ONLY for: editing files, submitting jobs, compiling code, quick file transfers
- **NEVER** run computations on the login node

For interactive work, request a compute node:
```bash
salloc -n1 -c2 --partition=public-interactive-cpu --time=1:00:00
```

## 3.3 File Transfer

```bash
# From local → HPC
scp local_file.py mibam:~/project/
rsync -avzP local_dir/ mibam:~/project/local_dir/

# From HPC → local
scp mibam:~/project/results.h5 ./
rsync -avzP mibam:~/project/results/ ./results/
```

In [ ]:
# Practice 3.1: Check your connection and environment
!echo "Hostname: $(hostname)"
!echo "User: $(whoami)"
!echo "Shell: $SHELL"
!echo "Home: $HOME"
!echo "Scratch: $(readlink -f ~/scratch 2>/dev/null || echo 'N/A')"
!echo "Kernel: $(uname -r)"
!echo "OS: $(cat /etc/os-release 2>/dev/null | grep PRETTY_NAME | cut -d= -f2)"
!echo "---"
!echo "Current SLURM jobs:"
!squeue -u $USER 2>/dev/null || echo '(not on a SLURM-managed node)'

---
# Module 4: Cluster Architecture
---

## 4.1 UNIGE HPC Clusters

The university operates three clusters:

| Cluster | Location | Interconnect | Public CPUs | Public GPUs |
|---------|----------|-------------|------------|------------|
| **Baobab** | Uni Dufour | IB 40 Gbit/s | ~900 | 0 |
| **Yggdrasil** | Observatory | IB 100 Gbit/s | ~3,000 | 44 |
| **Bamboo** | Campus Biotech | IB 100 Gbit/s | ~5,700 | 20 |

We use **Bamboo** for this course.

## 4.2 How a Cluster Works

```
                    ┌─────────────┐
  You (SSH) ──────→ │ Login Node  │  (2 CPUs, 8GB — editing/submitting only)
                    └──────┬──────┘
                           │ SLURM scheduler
                    ┌──────┴──────┐
              ┌─────┤  InfiniBand ├─────┐
              │     └─────────────┘     │
       ┌──────┴──────┐          ┌──────┴──────┐
       │ Compute     │   ...    │ Compute     │  (45+ CPU nodes, 11+ GPU nodes)
       │ Node cpu007 │          │ Node cpu052 │
       │ 128 cores   │          │ 64 cores    │
       │ 512 GB RAM  │          │ 256 GB RAM  │
       └──────┬──────┘          └──────┬──────┘
              │                        │
       ┌──────┴────────────────────────┴──────┐
       │          BeeGFS Parallel Storage      │
       │  /home (378TB SSD) + /scratch (shared)│
       └──────────────────────────────────────┘
```

## 4.3 Bamboo Partitions

| Partition | Time Limit | Nodes | Purpose |
|-----------|-----------|-------|---------|
| `debug-cpu` | 15 min | 2 | Quick tests |
| `public-cpu` | 4 days | 32 | Standard CPU jobs |
| `public-gpu` | 2 days | 2 | GPU jobs |
| `public-bigmem` | 4 days | 2 | High-memory jobs |
| `public-interactive-cpu` | 8 hours | 1 | Interactive sessions |
| `public-interactive-gpu` | 4 hours | 1 | Interactive GPU |
| `public-longrun-cpu` | 14 days | 1 | Long jobs (max 2 cores) |
| `public-short-cpu` | 1 hour | 2 | Very quick tests |
| `shared-cpu` | 12 hours | 45 | Shared (public + private nodes) |
| `shared-gpu` | 12 hours | 9 | Shared GPU |
| `shared-bigmem` | 12 hours | 2 | Shared high-memory |

## 4.4 CPU and GPU Models

**CPU**: AMD EPYC 7742 (128 cores), EPYC 72F3, EPYC 7763 — with 251–1024 GB RAM per node.

**GPU**: NVIDIA A100, H100/H200, RTX 4090/3090/3080/2080Ti, Titan RTX, RTX A6000/A5000.

In [ ]:
# Practice 4.1: Explore the cluster
!echo "=== Partition Info ==="
!sinfo -o "%20P %5a %10l %5D %20N" 2>/dev/null
!echo ""
!echo "=== Node Details (first 10) ==="
!sinfo -N -l 2>/dev/null | head -15
!echo ""
!echo "=== Currently Available CPUs ==="
!sinfo -o "%20P %10A" 2>/dev/null

---
# Module 5: Storage Systems
---

## 5.1 Storage Overview

| Location | Path | Quota | Backup | Speed | Purpose |
|----------|------|-------|--------|-------|---------|
| **Home** | `$HOME` (`/home/users/...`) | 1 TB | ✅ Daily | Medium | Code, configs, small data |
| **Scratch** | `$HOME/scratch` → `/srv/beegfs/scratch/...` | 10M files | ❌ None | Fast (BeeGFS) | Large data, temp results |
| **Fast** | `/srv/fast` | 500 GB | ❌ Erased at maintenance | Very fast (SSD) | Multi-node shared scratch |
| **Local** | `/scratch` (on compute node) | Node disk | ❌ Deleted after job | Fastest | Temporary job files |
| **Shared** | `/srv/share/users/...` | Per-node | ❌ Deleted when jobs end | Fast | Cross-job sharing on same node |

## 5.2 Key Rules

1. **Home is backed up** — store your code and important configs here
2. **Scratch is NOT backed up** — store regenerable data only
3. **Scratch has a retention policy**: files untouched for 3+ months are auto-deleted
4. **Never run `find`, `du`, or `ncdu`** on the root of scratch — it kills performance for everyone
5. **Keep < 1000 files per directory** for optimal I/O performance
6. **Conda environments** create thousands of tiny files — consider using containers instead

## 5.3 Optimizing I/O

For jobs with heavy I/O (many small reads/writes):
1. Copy data to **local node scratch** (`/scratch`) at job start
2. Process locally
3. Copy results back to BeeGFS at job end

Example from our NoisePy workflow — unpacking conda to node-local `/share`:
```bash
export ENV_DIR="/share/users/${SLURM_JOB_USER:0:1}/${SLURM_JOB_USER}/env"
tar -xzf ~/conda_pack/env.tar.gz -C "$ENV_DIR"
source "$ENV_DIR/bin/activate"
```

In [ ]:
# Practice 5.1: Check your storage
!echo "=== Home Directory ==="
!echo "Path: $HOME"
!du -sh ~/ 2>/dev/null | head -1
!echo ""
!echo "=== Scratch Directory ==="
!echo "Path: $(readlink -f ~/scratch 2>/dev/null)"
!echo ""
!echo "=== Disk Quotas ==="
!quota -s 2>/dev/null || echo "(quota command not available, try beegfs-get-quota-home-scratch.sh)"
!echo ""
!echo "=== Number of files in home ==="
!find ~/ -maxdepth 1 -not -path "*/scratch*" 2>/dev/null | wc -l

---
# Module 6: Environment Management — Modules & Conda
---

## 6.1 The Module System (Lmod)

Software on the cluster is managed through **Lmod** modules. This avoids version conflicts and keeps the environment clean.

| Command | Purpose |
|---------|---------|
| `module spider` | List all available software |
| `module spider python` | Find Python versions |
| `module load GCC/14.2.0` | Load a specific module |
| `module list` | Show loaded modules |
| `module purge` | Unload everything |
| `ml` | Shorthand for `module` |

### Toolchains

Software is built against specific compiler toolchains:
- **foss**: GCC + OpenMPI + OpenBLAS + FFTW (open-source, recommended)
- **intel**: Intel compilers + MKL + Intel MPI
- **fosscuda**: foss + CUDA (for GPU code)

To load a toolchain application, first load the base:
```bash
ml GCC/14.2.0 OpenMPI/5.0.7    # Load the toolchain
ml SciPy-bundle/2025.07         # Then the application
```

## 6.2 Conda Environments

Conda is widely used for Python-based scientific workflows:

```bash
# Load the base Anaconda module
ml Anaconda3/2024.02-1

# Create a new environment
conda create --name myenv python=3.11 numpy scipy obspy

# Activate it
conda activate myenv

# Install packages
conda install -c conda-forge pyasdf mpi4py
pip install noisepy
```

### Conda Environments in Your SLURM Scripts

```bash
#!/bin/bash
#SBATCH --partition=shared-cpu
#SBATCH --time=2:00:00

source ~/.bashrc          # Loads conda init
conda activate myenv      # Activate your environment

python my_script.py
```

> **Warning**: Conda environments create thousands of small files that stress BeeGFS. For production, consider using **conda-pack** (tar the env) or **Apptainer containers**.

## 6.3 Conda-Pack for MPI Jobs

When running MPI across multiple nodes, each node imports Python simultaneously, causing I/O storms on BeeGFS. Solution: pack your conda env into a tarball and unpack it to node-local storage:

```bash
# One-time: pack your environment
conda activate noisepy2
conda install conda-pack
conda-pack -o ~/conda_pack/noisepy_packed.tar.gz

# In your SLURM script: unpack to node-local /share
ENV_DIR="/share/users/${SLURM_JOB_USER:0:1}/${SLURM_JOB_USER}/noisepy_env"
tar -xzf ~/conda_pack/noisepy_packed.tar.gz -C "$ENV_DIR"
source "$ENV_DIR/bin/activate"
conda-unpack
```

This is exactly what our NoisePy production scripts do (see Module 15 of the ANT course).

## 6.4 Apptainer Containers (Advanced)

The most robust approach for reproducible environments:

```bash
# Pull a Docker image and convert to .sif
salloc --partition=shared-cpu --time=00:30:00 --cpus-per-task=4
apptainer pull docker://ubuntu:22.04

# Run inside the container
apptainer exec ubuntu_22.04.sif python3 -c "print('Hello from container')"
```

In [ ]:
# Practice 6.1: Explore modules
!echo "=== Currently loaded modules ==="
!module list 2>&1
!echo ""
!echo "=== Available Anaconda versions ==="
!module spider Anaconda3 2>&1 | head -20
!echo ""
!echo "=== Available CUDA versions ==="
!module spider CUDA 2>&1 | head -20
!echo ""
!echo "=== Your conda environments ==="
!conda env list 2>/dev/null || echo '(conda not initialized — run: source ~/.bashrc)'

---
# Module 7: SLURM — Job Submission Basics
---

## 7.1 What is SLURM?

**SLURM** (Simple Linux Utility for Resource Management) is the job scheduler. You tell it what resources you need, and it finds compute nodes for you.

Three ways to run jobs:

| Method | Script? | Blocking? | Arrays? | Use Case |
|--------|---------|-----------|---------|----------|
| `sbatch` | Yes | No | Yes | Production jobs |
| `salloc` | No | Yes | No | Interactive sessions |
| `srun` | No | Yes | No | Single commands |

## 7.2 Your First `sbatch` Script

```bash
#!/bin/bash
#SBATCH --job-name=my_first_job
#SBATCH --partition=debug-cpu
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=1000        # 1 GB
#SBATCH --time=00:15:00           # 15 minutes
#SBATCH --output="outslurm/%x-%j.out"

echo "Hello from $(hostname) at $(date)"
echo "Job ID: $SLURM_JOB_ID"
echo "Running on partition: $SLURM_JOB_PARTITION"

python -c "print('Python works!')"
```

Submit: `sbatch my_first_job.sh`

## 7.3 Key SBATCH Parameters

| Parameter | Example | Meaning |
|-----------|---------|---------|
| `--job-name` | `my_job` | Name shown in `squeue` |
| `--partition` | `shared-cpu` | Which partition to use |
| `--ntasks` | `1` | Number of MPI tasks |
| `--cpus-per-task` | `4` | Cores per task (for OpenMP/threading) |
| `--mem-per-cpu` | `10G` | Memory per core |
| `--mem` | `200G` | Total memory for the job |
| `--time` | `2-00:00:00` | 2 days wall time |
| `--output` | `out/%x-%j.out` | Stdout file (`%x`=name, `%j`=jobID) |
| `--mail-type` | `BEGIN,END,FAIL` | Email notifications |
| `--mail-user` | `you@email.com` | Your email |

## 7.4 Resource Types

### Single-threaded (most Python scripts)
```bash
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=4G
```

### Multi-threaded (NumPy/SciPy with OpenMP, multiprocessing)
```bash
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=20    # 20 threads
#SBATCH --mem-per-cpu=5G
```

### Distributed / MPI (NoisePy, parallel processing)
```bash
#SBATCH --ntasks=80           # 80 MPI ranks
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=10G
```

## 7.5 Interactive Sessions

```bash
# Quick interactive session (1 hour, 2 cores)
salloc -n1 -c2 --partition=public-interactive-cpu --time=1:00:00

# With X11 forwarding (for matplotlib plots)
salloc -n1 -c2 --partition=public-interactive-cpu --time=1:00:00 --x11

# Exit when done
exit
```

Common aliases (from your `.bashrc`):
```bash
alias salloc121="salloc -n1 -c2 --partition=public-interactive-cpu --time=1:00:00"
alias salloc163="salloc -n1 -c4 --partition=public-interactive-cpu --time=3:00:00"
```

In [ ]:
# Practice 7.1: Submit your first SLURM job
!mkdir -p ~/hpc_course/module7/outslurm && cat > ~/hpc_course/module7/hello_slurm.sh << 'SLURM'
#!/bin/bash
#SBATCH --job-name=hello_hpc
#SBATCH --partition=debug-cpu
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=1000
#SBATCH --time=00:05:00
#SBATCH --output="outslurm/%x-%j.out"

echo "============================================"
echo "Hello from the HPC!"
echo "============================================"
echo "Job ID:        $SLURM_JOB_ID"
echo "Job Name:      $SLURM_JOB_NAME"
echo "Partition:     $SLURM_JOB_PARTITION"
echo "Node:          $(hostname)"
echo "Cores:         $SLURM_CPUS_PER_TASK"
echo "Working Dir:   $(pwd)"
echo "Date:          $(date)"
echo "============================================"

# Show system info
echo "CPU Info:"
lscpu | grep "Model name"
echo "Memory:"
free -h | head -2

echo "============================================"
echo "Job completed successfully!"
SLURM

echo "--- Script ready ---"
cat ~/hpc_course/module7/hello_slurm.sh
echo ""
echo "Submit with: cd ~/hpc_course/module7 && sbatch hello_slurm.sh"
echo "Then check: squeue -u $USER"
echo "Then read output: cat outslurm/hello_hpc-<jobid>.out"
# Uncomment the next line to actually submit:
# !cd ~/hpc_course/module7 && sbatch hello_slurm.sh

---
# Module 8: SLURM — Job Arrays
---

## 8.1 Why Job Arrays?

Many scientific workflows are **embarrassingly parallel**: the same program runs independently on different inputs. Instead of submitting 1000 separate jobs, use a **job array**:

```bash
sbatch --array=0-999 my_job.sh
```

This creates 1000 jobs, each with a unique `$SLURM_ARRAY_TASK_ID` (0 to 999).

## 8.2 Array Job Patterns

### Pattern 1: Index into a file list
```bash
#SBATCH --array=1-185

fname="/path/to/station_list.csv"
ROW=${SLURM_ARRAY_TASK_ID}
sfile=$(awk -F, "NR==$ROW {print \$0}" $fname)
python process.py "$sfile"
```

### Pattern 2: Index into an array of parameters
```bash
#SBATCH --array=0-19

periods=(0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0 1.1 1.2)
current_period=${periods[$SLURM_ARRAY_TASK_ID]}
python 2D_v2.py $current_period
```

### Pattern 3: Process N items per task (batch processing)
```bash
#SBATCH --array=0-40

fname="/path/to/file_list.txt"
for k in $(seq 0 99); do
    ROW=$(( ${SLURM_ARRAY_TASK_ID} * 100 + $k ))
    sfile=$(sed -n "${ROW}p" $fname)
    python dispersion_curves_V2.py $sfile
done
```

## 8.3 Throttling

Limit concurrent tasks to avoid overwhelming the cluster:
```bash
#SBATCH --array=0-999%50    # Max 50 running simultaneously
#SBATCH --array=1-352%300   # Max 300 at once
```

## 8.4 Output Files

Use `%a` for array task ID in output filenames:
```bash
#SBATCH --output="outslurm/%x_%a.out"    # e.g., myjob_42.out
#SBATCH --output="outslurm/%x-%A_%a.out" # e.g., myjob-123456_42.out
```

- `%A` = array job ID (same for all tasks)
- `%a` = array task ID (unique per task)
- `%j` = job ID (unique per task, different from %a)

## 8.5 Real Examples from Our Workflows

### HVSR processing (185 stations)
```bash
#SBATCH --array=1-185
fname="vulcano_stations.csv"
ROW=${SLURM_ARRAY_TASK_ID}
sfile=$(awk -F, "NR==$ROW {print \$0}" $fname)
python -u hvsr_oneday.py $sfile dfa
```

### BayHunter MCMC (352 grid cells, 30 chains each)
```bash
#SBATCH --array=1-352%300
#SBATCH --cpus-per-task=30
dispfile=$(find $path -name 'disp_*.dat' | sort | sed -n "${SLURM_ARRAY_TASK_ID}p")
python run_mcmc.py "$dispfile"
```

### MANgOSTA tomography (one period per task)
```bash
#SBATCH --array=4-4
plist=(0.9 1.0 1.1 ... 5.0)
PER=${plist[$SLURM_ARRAY_TASK_ID]}
# Build config file dynamically, then run
python mangosta_tomo.py tomo_${SLURM_ARRAY_TASK_ID}.in
```

In [ ]:
# Practice 8.1: Create an array job
!mkdir -p ~/hpc_course/module8/outslurm && cat > ~/hpc_course/module8/array_demo.sh << 'SLURM'
#!/bin/bash
#SBATCH --job-name=array_demo
#SBATCH --partition=debug-cpu
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=500
#SBATCH --time=00:05:00
#SBATCH --output="outslurm/%x_%a.out"
#SBATCH --array=0-4

# Define parameters as an array
VELOCITIES=(1.5 2.0 2.5 3.0 3.5)

# Get the velocity for this task
VEL=${VELOCITIES[$SLURM_ARRAY_TASK_ID]}

echo "Array Task ID: $SLURM_ARRAY_TASK_ID"
echo "Processing velocity: $VEL km/s"
echo "Node: $(hostname)"
echo "Time: $(date)"

# Simulate some computation
python3 -c "
import time
vel = $VEL
wavelength = vel * 1.0  # 1 second period
print(f'Velocity: {vel} km/s')
print(f'Wavelength at 1s period: {wavelength} km')
print(f'Depth sensitivity: ~{wavelength/3:.1f} km')
time.sleep(2)
print('Done!')
"
SLURM

echo "Script ready at ~/hpc_course/module8/array_demo.sh"
echo "Submit with: cd ~/hpc_course/module8 && sbatch array_demo.sh"
echo "This will create 5 jobs, one for each velocity."
cat ~/hpc_course/module8/array_demo.sh

---
# Module 9: SLURM — Advanced Features
---

## 9.1 Job Dependencies

Chain jobs so that step 2 starts only after step 1 completes:

```bash
# Submit step 1
JOB1=$(sbatch --parsable step1_mcmc.slurm)
echo "Step 1 submitted: job $JOB1"

# Submit step 2, waiting for step 1
JOB2=$(sbatch --parsable --dependency=aftercorr:$JOB1 step2_postprocess.slurm)
echo "Step 2 submitted: job $JOB2 (waits for $JOB1)"
```

### Dependency Types

| Type | Meaning |
|------|---------|
| `afterok:JOBID` | Start after job succeeds (exit 0) |
| `afterany:JOBID` | Start after job finishes (any exit code) |
| `afternotok:JOBID` | Start after job fails |
| `aftercorr:JOBID` | For arrays: each task starts after its corresponding task |
| `singleton` | Only one job of this name runs at a time |

### Real Example: BayHunter Pipeline
```bash
#!/bin/bash
# submit.sh — chain MCMC + post-processing
JOB1=$(sbatch --parsable step1_mcmc.slurm)
JOB2=$(sbatch --parsable --dependency=aftercorr:$JOB1 step2_postprocess.slurm)
```
With `aftercorr`, array task 42 of step 2 starts as soon as array task 42 of step 1 finishes — no need to wait for the entire array.

## 9.2 Checkpointing

For long computations that exceed partition time limits:

1. Save intermediate state to a checkpoint file
2. Submit the job with shorter wall time
3. When time runs out, restart from the checkpoint

```python
# In your Python script
import pickle, os

checkpoint_file = f"checkpoint_{os.environ.get('SLURM_ARRAY_TASK_ID', 0)}.pkl"

# Load checkpoint if exists
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'rb') as f:
        state = pickle.load(f)
    start_iter = state['iteration']
else:
    start_iter = 0
    state = {'iteration': 0, 'results': []}

# Main loop
for i in range(start_iter, 100000):
    # ... computation ...
    state['iteration'] = i
    if i % 1000 == 0:  # Save every 1000 iterations
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(state, f)
```

## 9.3 Priority and Scheduling

SLURM priority is determined by four factors:

| Factor | Weight | How to Improve |
|--------|--------|---------------|
| **Partition** | 15,000 (public) / 3,750 (shared) | Choose the right partition |
| **Fairshare** | 30,000 | Use less → higher priority (resets every 2 weeks) |
| **Job size** | 1,000 | Larger jobs get slight boost |
| **Age** | 300 | Waiting longer increases priority |

Check your job's priority: `sprio -j JOBID`

### Backfill

SLURM uses **backfill** scheduling: smaller/shorter jobs can jump ahead of larger queued jobs if they fit in the gaps. This means:
- **Accurate time estimates help everyone** — a 1-hour job can backfill where a 4-day job can't
- Overestimating time wastes your priority and blocks backfill

## 9.4 GPU Jobs

```bash
#!/bin/bash
#SBATCH --partition=shared-gpu
#SBATCH --gpus=1                  # Request 1 GPU
#SBATCH --time=01:00:00

module load CUDA/12.6.0
echo "GPU assigned: $CUDA_VISIBLE_DEVICES"
nvidia-smi
python my_gpu_script.py
```

Request specific GPU types:
```bash
#SBATCH --gpus=titan:1     # Specific type
```

---
# Module 10: MPI and Parallel Computing
---

## 10.1 Types of Parallelism

| Type | SLURM Params | Technology | Example |
|------|-------------|-----------|---------|
| **Single-threaded** | `--ntasks=1 --cpus=1` | None | Simple Python script |
| **Multi-threaded** | `--ntasks=1 --cpus=N` | OpenMP, multiprocessing | NumPy with MKL, BayHunter chains |
| **Distributed** | `--ntasks=N --cpus=1` | MPI (mpi4py) | NoisePy S0/S1/S2 |
| **Hybrid** | `--ntasks=N --cpus=M` | MPI + OpenMP | Large simulations |

## 10.2 MPI with NoisePy

Our NoisePy workflow is the prime example of MPI-parallel processing:

```bash
#SBATCH --ntasks=80           # 80 MPI ranks
#SBATCH --cpus-per-task=1     # 1 core per rank
#SBATCH --mem-per-cpu=10G

mpiexec -n $SLURM_NTASKS python S1_fft_cc_MPI.py S1_params.yaml
```

How it works:
- **S0**: Each MPI rank downloads/converts different time chunks
- **S1**: Each rank processes different `.h5` files (FFT + cross-correlation)
- **S2**: Each rank handles different station pairs (stacking)

### Choosing `ntasks`

| Step | Max Useful ntasks |
|------|-------------------|
| S0 | Number of time chunks (total hours / `inc_hours`) |
| S1 | Number of `.h5` files in `DATADIR` |
| S2 | Number of station pairs: N×(N-1)/2 |

## 10.3 Multi-threaded Jobs (BayHunter, BayesBay)

Some tools use Python's `multiprocessing` or C-level threading instead of MPI:

```bash
#SBATCH --ntasks=1            # Single process
#SBATCH --cpus-per-task=30    # 30 threads/chains
#SBATCH --mem-per-cpu=15G

python run_mcmc.py "$dispfile"
```

BayHunter runs 30 MCMC chains in parallel — each chain is a thread, not an MPI rank.

## 10.4 Conda-Pack for Multi-Node MPI

When MPI spans multiple nodes, all nodes import Python simultaneously. This creates an I/O storm on BeeGFS. Our production solution:

```bash
# In SLURM script: unpack conda to node-local /share (once per node)
srun --ntasks=$SLURM_NNODES --ntasks-per-node=1 bash -c '
    ENV_DIR="/share/users/${SLURM_JOB_USER:0:1}/${SLURM_JOB_USER}/env"
    mkdir -p "$ENV_DIR"
    tar -xzf ~/conda_pack/noisepy_packed.tar.gz -C "$ENV_DIR"
    source "$ENV_DIR/bin/activate"
    conda-unpack
'
source "$ENV_DIR/bin/activate"
mpiexec -n $SLURM_NTASKS python S1_fft_cc_MPI.py S1_params.yaml
```

This eliminates the `FileNotFoundError` race condition when many MPI ranks import `obspy` simultaneously from BeeGFS.

---
# Module 11: GPU Computing
---

## 11.1 GPU Partitions on Bamboo

| Partition | Time Limit | GPUs Available |
|-----------|-----------|---------------|
| `public-gpu` | 2 days | 2 nodes |
| `public-interactive-gpu` | 4 hours | 1 node |
| `shared-gpu` | 12 hours | 9 nodes |

## 11.2 Requesting GPUs

```bash
#SBATCH --partition=shared-gpu
#SBATCH --gpus=1                    # 1 GPU (any type)
#SBATCH --gpus=titan:2              # 2 Titan GPUs
#SBATCH --gpus=2,VramPerGpu:40G     # 2 GPUs with ≥40GB VRAM
```

## 11.3 GPU Job Template

```bash
#!/bin/bash
#SBATCH --job-name=gpu_test
#SBATCH --partition=public-interactive-gpu
#SBATCH --gpus=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=01:00:00
#SBATCH --output="outslurm/%x-%j.out"

module load CUDA/12.6.0

echo "GPU devices: $CUDA_VISIBLE_DEVICES"
nvidia-smi

python -c "
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
"
```

## 11.4 Common GPU Issues

- **CUDA version mismatch**: Load the CUDA module that matches your PyTorch/TensorFlow build
- **Out of memory**: Reduce batch size or use gradient checkpointing
- **`Illegal instruction`**: Force compatible CPU instructions:
  ```bash
  export OPENBLAS_CORETYPE=Nehalem
  export MKL_CBWR=COMPATIBLE
  ```

---
# Module 12: Job Monitoring and Debugging
---

## 12.1 Essential Monitoring Commands

| Command | Purpose | Example |
|---------|---------|---------|
| `squeue -u $USER` | Your running/pending jobs | `squeue -u henrymi` |
| `squeue --start` | Estimated start times | For pending jobs |
| `scontrol show job JOBID` | Full job details | Partition, node, memory |
| `sacct -j JOBID` | Completed job history | CPU time, memory, exit code |
| `seff JOBID` | Efficiency report | CPU%, memory% utilization |
| `sinfo` | Partition status | Available nodes/cores |
| `pestat` | Detailed node usage | Load, memory, jobs per node |
| `sprio -j JOBID` | Priority breakdown | Fairshare, age, size, partition |

## 12.2 Checking Job Output

```bash
# While job is running — watch the log live
tail -f outslurm/myjob-123456.out

# After job completes — check efficiency
seff 123456

# Example seff output:
# Job ID: 123456
# Cores per node: 20
# CPU Utilized: 18:32:45
# CPU Efficiency: 92.71% of 20:00:00 core-walltime
# Memory Utilized: 15.23 GB
# Memory Efficiency: 76.15% of 20.00 GB
```

## 12.3 Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| `OUT_OF_MEMORY` | Memory exceeded | Increase `--mem-per-cpu` or `--mem` |
| `TIMEOUT` | Wall time exceeded | Increase `--time` or add checkpointing |
| `FAILED` (exit 1) | Script error | Check output log for Python traceback |
| Job stuck `PENDING` | No resources available | Check `squeue --start` or use different partition |
| `NODE_FAIL` | Hardware issue | Resubmit (transient) |
| `FileNotFoundError` | BeeGFS race condition | Use conda-pack (Module 10) |

In [ ]:
# Practice 12.1: Monitor your jobs
!echo "=== Your Current Jobs ==="
!squeue -u $USER 2>/dev/null || echo "No jobs running"
!echo ""
!echo "=== Recent Job History (last 5) ==="
!sacct --format=JobID,JobName,Partition,State,Elapsed,MaxRSS,ExitCode -u $USER 2>/dev/null | head -10
!echo ""
!echo "=== Partition Availability ==="
!sinfo -o "%20P %5a %10l %10A" 2>/dev/null

## 12.4 Real Issues from the HPC Community Forum

The [HPC Community Forum](https://hpc-community.unige.ch/) is a Discourse-based platform where UNIGE users report issues, request software, and share solutions. Below are the most commonly reported problems and their fixes, drawn directly from the forum.

### BeeGFS Scratch Failures
**Symptom**: Jobs crash with I/O errors, `FileNotFoundError`, or `Permission denied` on `/srv/beegfs/scratch/`.
**Forum**: 15+ replies — the most discussed issue on the forum.
**Workarounds**:
- Copy critical input data to node-local `/scratch` or `/tmp` at job start
- Use `conda-pack` to avoid concurrent reads from BeeGFS (see Module 10)
- If scratch is completely down, wait for the HPC team to restore it
- Check [HPC Status Page](https://hpc-community.unige.ch/) for announcements

### BeeGFS v8 Quota Access
**Symptom**: `beegfs-get-quota-home-scratch.sh` returns errors after BeeGFS upgrade to v8.
**Fix**: The HPC team posted a temporary workaround — check the ChangeLog category on the forum.

### Package File Visibility in Concurrent Jobs
**Symptom**: R/Pixi/conda packages intermittently "not found" when many array tasks start simultaneously.
**Cause**: BeeGFS metadata caching lag when hundreds of tasks read the same conda env directory.
**Fix**: Use `conda-pack` (Module 6.3) or copy the environment to node-local `/share` storage before activating.

### PMIx Error: "not-found"
**Symptom**: MPI jobs fail with `PMIx error: not-found` on Bamboo.
**Fix**: Ensure you load matching GCC + OpenMPI versions. For example: `ml GCC/14.2.0 OpenMPI/5.0.7`. Mismatched toolchain versions cause PMIx initialization failures. Also reported: `GCC/12.3.0` + `OpenMPI/4.1.5` broken on `debug-cpu` — use newer versions.

### GPU Jobs Not Starting Despite Idle Nodes
**Symptom**: `squeue` shows your GPU job pending, but `sinfo` shows idle GPU nodes.
**Forum**: Reported for both Bamboo and Baobab, including specific node failures (e.g., `gpu008`, `gpu027`, `gpu048`).
**Cause**: GPU type/VRAM constraints, faulty GPUs taken offline, or memory requirements exceeding available node resources.
**Fix**: Check `scontrol show job JOBID` for the `Reason` field. Try `--gpus=1` without specifying type. If a specific GPU node is faulty, exclude it: `--exclude=gpu008`.

### SIGTERM Without Reason
**Symptom**: Jobs terminated with SIGTERM before reaching the time limit.
**Cause**: Node issues, preemption, or OOM kills not properly reported.
**Fix**: Check `sacct -j JOBID --format=JobID,State,ExitCode,DerivedExitCode,Comment`. Add memory headroom.

### VS Code Remote-SSH Hanging on Bamboo
**Symptom**: VS Code Remote-SSH connection hangs indefinitely when connecting to `login1.bamboo`.
**Workaround**: Use terminal SSH + vim/nano, or use **OpenOnDemand** for a web-based IDE experience. The forum suggests the issue is related to VS Code's background processes exceeding login node limits.

### Matplotlib / Logging Crashes After Maintenance
**Symptom**: Jobs that worked before a maintenance window now crash with logging or matplotlib errors.
**Fix**: Rebuild or re-pack your conda environment after major system updates. Copy matplotlib data to local scratch (see Module 15.6 pattern).

### "Kinit" Not Found on Compute Nodes
**Symptom**: `kinit: command not found` when trying to use Kerberos auth on compute nodes.
**Context**: Kerberos tools are not available on all compute nodes. Use SSH key-based authentication instead.

### Very Slow Login and Compute Nodes
**Symptom**: Everything is sluggish — `ls` takes seconds, `module load` hangs.
**Cause**: Usually a BeeGFS metadata server overload (someone running `find` on all of scratch, or a storage failure).
**Fix**: Wait and check the status page. Report to `hpc@unige.ch` if persistent.

---

### Software Wish List (from forum)

The forum has a "Wish List" category where users request software installations. Recent installs include:
- **R 4.5.2**, **ORCA 6.1.1**, **Nextflow 26.04.6**
- **SimNIBS 4.6.0**, **macs3 3.0.3**, **dorado 2.0.1**
- **Code-server 4.105.1**, **jupyterlmod 5.3.0**
- **LLM deployment** feasibility discussed for Bamboo (LMStudio now available via OpenOnDemand)

To request new software: post in the [Wish List category](https://hpc-community.unige.ch/) or email `hpc@unige.ch`.

---

### How to Contact the HPC Team

When emailing `hpc@unige.ch`, always include:
1. Your **username** and **cluster** (Bamboo/Baobab/Yggdrasil)
2. The **job ID** (`$SLURM_JOB_ID`)
3. The **exact error message** (copy-paste from the log, not a screenshot)
4. The **SLURM script** you submitted
5. The **output log** file path on the cluster

> **Tip**: The community forum is also a great place to search before emailing — your issue may already have a solution posted.

---
# Module 13: Data Management and Transfer
---

## 13.1 rsync — The Essential Transfer Tool

`rsync` is the best tool for transferring data between clusters or between your laptop and the HPC:

```bash
# Local → HPC
rsync -avzP local_data/ mibam:~/scratch/project/data/

# HPC → HPC (between clusters)
rsync -aviuzPrg /path/on/bamboo/ yggdrasil:/path/on/ygg/

# Key flags:
#   -a  Archive mode (preserves everything)
#   -v  Verbose
#   -z  Compress during transfer
#   -P  Show progress + resume partial transfers
#   -u  Skip files newer at destination (safe re-runs)
#   -n  Dry run (preview only)
```

Always do a **dry run** first: `rsync -avzPn source/ dest/`

## 13.2 rclone — Cloud Storage Transfer

For transferring data from cloud storage (SWITCHdrive, Google Drive, S3, etc.):

```bash
#!/bin/bash
#SBATCH --job-name=rclone_download
#SBATCH --partition=public-cpu
#SBATCH --time=3-00:00:00

module load rclone/1.72.0

rclone copy swb:/remote_folder /path/to/local \
    --transfers 4 --checkers 4 \
    --retries 3 --stats 5m -v

# Verify transfer
rclone check swb:/remote_folder /path/to/local --size-only
```

## 13.3 Data Lifecycle

```
1. Raw data arrives (download/transfer)     → scratch/RAW_DATA/
2. Preprocessing (S0: clean + convert)      → scratch/CLEAN_DATA/
3. Processing (S1: CC, S2: stack)           → scratch/CCF/, scratch/STACK/
4. Analysis (dispersion, tomography)        → scratch/results/
5. Final results → copy to home or archive
6. Delete intermediate files from scratch!
```

> **Remember**: Scratch files untouched for 3 months are auto-deleted. Move important results to `$HOME` or archive.

---
# Module 14: Best Practices — Being a Good HPC Citizen
---

## 14.1 The Golden Rules

1. **Never compute on the login node** — use `salloc` or `sbatch`
2. **Request only what you need** — overallocation wastes resources for everyone
3. **Test with `debug-cpu` first** — 15 minutes is enough to catch most bugs
4. **Estimate resources accurately** — check with `seff` after the first run
5. **Clean up scratch** — delete old files, don't hoard data

## 14.2 Resource Estimation Workflow

```
1. Run a small test on debug-cpu (15 min)
2. Check seff → see actual CPU/memory usage
3. Scale up: if test used 2GB for 10 files, and you have 1000 files → request ~3GB
4. Submit to shared-cpu (12h) or public-cpu (4 days)
5. After first real run, check seff again and adjust
```

## 14.3 Common Mistakes

| Mistake | Impact | Fix |
|---------|--------|-----|
| Requesting 40 cores for a single-threaded Python script | 39 cores sit idle | Use `--cpus-per-task=1` |
| Requesting 500 GB memory for a 5 GB job | Blocks entire node | Start small, scale up |
| Setting `--time=4-00:00:00` for a 1-hour job | Blocks backfill scheduling | Estimate accurately |
| Running `conda install` on login node | Slow + I/O heavy | Use `salloc` first |
| Storing 1M tiny files on scratch | Kills metadata performance | Tar/archive small files |
| Not making `outslurm/` directory | Job fails silently | `mkdir -p outslurm` |

## 14.4 Environment Best Practices

```bash
# In your SLURM scripts, always:
source ~/.bashrc               # Initialize conda
conda activate myenv           # Activate environment

# For matplotlib on headless nodes:
export MPLBACKEND=Agg          # Non-interactive backend

# For CPU compatibility across node types:
export OPENBLAS_CORETYPE=Nehalem
export MKL_CBWR=COMPATIBLE
```

## 14.5 Think Green

Bamboo consumes ~80 kW (excluding cooling). Every wasted core-hour has a real environmental cost:
- Right-size your jobs → reduce power consumption
- Delete unnecessary data → reduce storage overhead
- Use efficient algorithms → less compute time

The target: **~80% CPU utilization** cluster-wide.

---
# Module 15: Real-World HPC Workflows
---

This module walks through our actual production workflows. Each represents a different parallelization pattern.

## 15.1 NoisePy — MPI Distributed Processing

**Pattern**: MPI parallelism across time chunks / station pairs

```bash
#!/bin/bash
#SBATCH --job-name=S1_noisepy
#SBATCH --partition=public-cpu
#SBATCH --ntasks=80
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu=10G
#SBATCH --time=2-00:00:00
#SBATCH --output="outslurm/%x-%j.out"
#SBATCH --mail-type=BEGIN,END
#SBATCH --mail-user=your.email@unige.ch

# Stage conda env to node-local storage (avoid BeeGFS race conditions)
ENV_TAR="$HOME/conda_pack/noisepy_packed.tar.gz"
ENV_DIR="/share/users/${SLURM_JOB_USER:0:1}/${SLURM_JOB_USER}/noisepy_env"
srun --ntasks=$SLURM_NNODES --ntasks-per-node=1 bash -c '
    mkdir -p "$ENV_DIR" && tar -xzf "$ENV_TAR" -C "$ENV_DIR"
    source "$ENV_DIR/bin/activate" && conda-unpack
'
source "$ENV_DIR/bin/activate"

PARAMS="S1_params.yaml"
mpiexec -n $SLURM_NTASKS python S1_fft_cc_MPI.py $PARAMS
```

## 15.2 BayHunter — Array Jobs with Dependencies

**Pattern**: Job arrays (one task per grid cell) + dependency chaining

```bash
# submit.sh
JOB1=$(sbatch --parsable step1_mcmc.slurm)       # Array: 1-352, 30 cpus each
JOB2=$(sbatch --parsable --dependency=aftercorr:$JOB1 step2_postprocess.slurm)
```

## 15.3 MANgOSTA — Dynamic Config Generation

**Pattern**: Array job where each task generates its own config file

```bash
#SBATCH --array=0-41
plist=(0.9 1.0 1.1 ... 5.0)
PER=${plist[$SLURM_ARRAY_TASK_ID]}

# Dynamically build the input config
echo "PERIODS ${PER}" > tomo_${SLURM_ARRAY_TASK_ID}.in
echo "INVERSION LINEAR" >> tomo_${SLURM_ARRAY_TASK_ID}.in
echo "MULTISCALE 7" >> tomo_${SLURM_ARRAY_TASK_ID}.in
# ...
python mangosta_tomo.py tomo_${SLURM_ARRAY_TASK_ID}.in
```

## 15.4 BayesBay 2D — Per-Period Inversion

**Pattern**: Array job over periods + CPU workarounds

```bash
#SBATCH --array=0-19
#SBATCH --cpus-per-task=3
#SBATCH --mem=5G

# Fix CPU instruction set mismatch across heterogeneous nodes
export OPENBLAS_CORETYPE=Nehalem
export MKL_CBWR=COMPATIBLE
export MPLBACKEND=Agg

periods=(0.1 0.2 0.3 ... 1.2)
python -u 2D_v2.py ${periods[$SLURM_ARRAY_TASK_ID]}
```

## 15.5 Dispersion Picking — Batch Array Processing

**Pattern**: Each array task processes 100 station pairs

```bash
#SBATCH --array=0-40
fname="stack_files.txt"
for k in $(seq 0 99); do
    ROW=$(( ${SLURM_ARRAY_TASK_ID} * 100 + $k ))
    sfile=$(sed -n "${ROW}p" $fname)
    python dispersion_curves_V2.py $sfile
done
```

## 15.6 Joint HV-DC Inversion — Complex Environment Setup

**Pattern**: Array job with matplotlib workaround for parallel startup

```bash
#SBATCH --cpus-per-task=40
ml GCC/14.2.0 OpenMPI/5.0.7

# Copy matplotlib data to node-local scratch to avoid NFS contention
LOCAL_MPL=/tmp/${USER}_${SLURM_JOB_ID}_${SLURM_ARRAY_TASK_ID}/mpl-data
cp -r "$(python -c 'import matplotlib; print(matplotlib.get_data_path())')" "$LOCAL_MPL"
export MATPLOTLIBDATA="$LOCAL_MPL"
export MPLCONFIGDIR=/tmp/${USER}_${SLURM_JOB_ID}_${SLURM_ARRAY_TASK_ID}/mplconfig

python -u joint_inversion_hv_disp_v2.py "$HV_FILE" "$DISP_FILE"

# Clean up
rm -rf "/tmp/${USER}_${SLURM_JOB_ID}_${SLURM_ARRAY_TASK_ID}"
```

---
# Module 16: Putting It All Together — End-to-End Exercise
---

## 16.1 Exercise: Complete ANT Workflow on the HPC

Build and submit a complete ambient noise tomography pipeline:

### Step 1: Setup
```bash
mkdir -p ~/hpc_course/project/{scripts,params,outslurm}
cd ~/hpc_course/project
```

### Step 2: Write the YAML configuration
Create `params/S1_params.yaml` with appropriate settings for your dataset.

### Step 3: Write the SLURM submission scripts
- `scripts/step0_convert.slurm` — Convert raw data to ASDF
- `scripts/step1_cc.slurm` — Cross-correlations
- `scripts/step2_stack.slurm` — Stacking
- `scripts/step3_disp.slurm` — Dispersion picking (array job)

### Step 4: Write the orchestrator
```bash
#!/bin/bash
# scripts/run_all.sh — Submit the full pipeline with dependencies

JOB0=$(sbatch --parsable scripts/step0_convert.slurm)
echo "S0 submitted: $JOB0"

JOB1=$(sbatch --parsable --dependency=afterok:$JOB0 scripts/step1_cc.slurm)
echo "S1 submitted: $JOB1 (after S0)"

JOB2=$(sbatch --parsable --dependency=afterok:$JOB1 scripts/step2_stack.slurm)
echo "S2 submitted: $JOB2 (after S1)"

JOB3=$(sbatch --parsable --dependency=afterok:$JOB2 scripts/step3_disp.slurm)
echo "S3 submitted: $JOB3 (after S2)"

echo "Full pipeline submitted! Monitor with: squeue -u $USER"
```

### Step 5: Monitor
```bash
squeue -u $USER              # Check job status
tail -f outslurm/*.out       # Watch logs
sacct -j $JOB_ID             # Check completed jobs
seff $JOB_ID                 # Resource efficiency
```

## 16.2 Checklist

Before submitting a production job, verify:

- [ ] Script tested on `debug-cpu` partition first
- [ ] `outslurm/` directory exists
- [ ] Correct conda environment specified
- [ ] `--mem` or `--mem-per-cpu` is set
- [ ] `--time` is realistic (check with `seff` from test run)
- [ ] `--ntasks` and `--cpus-per-task` match your parallelism type
- [ ] Output paths exist and are writable
- [ ] Input data paths are correct (no typos!)
- [ ] `export MPLBACKEND=Agg` if using matplotlib

## 16.3 Quick Reference Card

```
# Submit a job
sbatch my_script.sh

# Check your jobs
squeue -u $USER

# Cancel a job
scancel JOBID
scancel -u $USER --state=pending    # Cancel all pending

# Interactive session
salloc -n1 -c2 --partition=public-interactive-cpu --time=1:00:00

# Job efficiency (after completion)
seff JOBID

# Partition info
sinfo -o "%20P %5a %10l %10A"

# Recent job history
sacct --format=JobID,JobName,State,Elapsed,MaxRSS -u $USER

# Disk quota
beegfs-get-quota-home-scratch.sh

# Module system
module spider KEYWORD
module load MODULE/VERSION
module list
module purge
```

---

**Congratulations!** You now have the skills to effectively use the UNIGE HPC for any computational workflow.

For questions and community support:
- 📧 hpc@unige.ch
- 🌐 https://hpc-community.unige.ch/
- 📖 https://doc.eresearch.unige.ch/hpc/toc